# 10 · LGWR / GWR

Template para el modelo geoestadístico local. En el repo actual la implementación disponible es `GWRModel`, así que este notebook queda preparado para usar ese wrapper y analizar coeficientes locales.

## Hipótesis del modelo

- La relación entre features y `precio_m2` cambia según la localización.
- El modelo debería capturar heterogeneidad espacial mejor que una regresión global.
- La interpretabilidad principal viene por mapas de coeficientes locales y diagnóstico de residuos.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from ml_core.models.gwrmodel import GWRModel

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "output" / "10_lgwr"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

## Datos y configuración

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "rental_listings_model_input.parquet"
# df = pd.read_parquet(DATA_PATH)

target_col = "price_m2"
coord_cols = ["lon", "lat"]
feature_cols = []

# Cargar o reconstruir exactamente el mismo split definido en 01_baseline_and_protocol.

## Entrenamiento

In [ ]:
gwr_params = {
    "kernel": "bisquare",
    "fixed": False,
}

# X_train = train_df[feature_cols]
# y_train = train_df[target_col]
# coords_train = train_df[coord_cols].to_numpy()
# model = GWRModel(gwr_params=gwr_params)
# model.fit(X_train, y_train, coords_train)
# model

## Tuning

Podés barrer `kernel`, `fixed`, `bw` y distintas definiciones de features. Si hacés tuning, dejá el criterio registrado en `run_config.json`.

In [ ]:
# model.tune_hyperparameters(X_train, y_train.values.reshape(-1, 1), coords_train)
# best_config = {"gwr_params": gwr_params, "selected_bw": getattr(model, "bw_", None)}
# best_config

## Evaluación global

In [ ]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    residuals = y_true - y_pred
    return {
        "rmse": float(np.sqrt(np.mean(residuals ** 2))),
        "mae": float(np.mean(np.abs(residuals))),
        "r2": float(1 - np.sum(residuals ** 2) / np.sum((y_true - y_true.mean()) ** 2)),
        "bias": float(residuals.mean()),
    }

# X_test = test_df[feature_cols]
# coords_test = test_df[coord_cols].to_numpy()
# y_test = test_df[target_col]
# y_pred = model.predict(X_test, coords_test)
# metrics = regression_metrics(y_test, y_pred)
# metrics

## Diagnóstico espacial

In [ ]:
# results = test_df[[target_col] + coord_cols].copy()
# results["y_pred"] = np.asarray(y_pred).reshape(-1)
# results["residual"] = results[target_col] - results["y_pred"]
# results.head()

## Interpretación

En GWR la interpretación más útil no suele ser una `feature importance` global, sino:

- coeficientes locales
- estabilidad espacial de cada feature
- signos dominantes por zona
- residuos espacialmente agrupados

In [ ]:
# local_betas = pd.DataFrame(model.results_.params, columns=["intercept"] + feature_cols)
# local_betas = pd.concat([train_df[coord_cols].reset_index(drop=True), local_betas], axis=1)
# local_betas.head()

## Export de artefactos

In [ ]:
# test_export = test_df[[target_col] + coord_cols].copy()
# test_export = test_export.rename(columns={target_col: "y_true"})
# test_export["y_pred"] = np.asarray(y_pred).reshape(-1)
# test_export["residual"] = test_export["y_true"] - test_export["y_pred"]
# test_export["split"] = "test"
# test_export.to_parquet(OUTPUT_DIR / "test_predictions.parquet", index=False)
# local_betas.to_parquet(OUTPUT_DIR / "interpretability.parquet", index=False)
# (OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False))
# (OUTPUT_DIR / "run_config.json").write_text(json.dumps(best_config, indent=2, ensure_ascii=False))